In [0]:
from pyspark.sql.functions import sum, countDistinct, round, col, avg, date_format

In [0]:
spark.table("adbrag.project4_schema.gold_product_year_sales") \
    .groupBy("ProductName") \
    .sum("TotalSalesAmount") \
    .orderBy(col("sum(TotalSalesAmount)").desc()) \
    .limit(5) \
    .display()

In [0]:
spark.table("adbrag.project4_schema.gold_customer_sales") \
    .orderBy(col("TotalSalesAmount").desc()) \
    .limit(5) \
    .display()

In [0]:
spark.table("adbrag.project4_schema.gold_yearly_summary") \
    .orderBy("SourceYear") \
    .display()

In [0]:
import plotly.express as px

In [0]:
df_yearly = spark.table("adbrag.project4_schema.gold_yearly_summary")
pdf_yearly = df_yearly.orderBy("SourceYear").toPandas()

fig = px.bar(
    pdf_yearly,
    x="SourceYear",
    y="TotalSalesAmount",
    text="TotalSalesAmount",
    title="Yearly Total Sales Amount"
)

fig.update_traces(textposition="outside")
fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Total Sales Amount",
    uniformtext_minsize=8,
    uniformtext_mode="hide"
)

fig.show()

In [0]:
df_monthly = spark.table("adbrag.project4_schema.gold_monthly_sales")
pdf_monthly = df_monthly.orderBy("OrderMonth").toPandas()

fig = px.line(
    pdf_monthly,
    x="OrderMonth",
    y="TotalSalesAmount",
    markers=True,
    title="Monthly Sales Trend"
)

fig.update_layout(
    xaxis_title="Order Month",
    yaxis_title="Total Sales Amount"
)

fig.show()

In [0]:
df_top_products = (
    spark.table("adbrag.project4_schema.gold_product_year_sales")
    .groupBy("ProductName")
    .agg(sum("TotalSalesAmount").alias("GrandTotalSales"))
    .orderBy(col("GrandTotalSales").desc())
    .limit(10)
)

pdf_top_products = df_top_products.toPandas()

fig = px.bar(
    pdf_top_products,
    x="GrandTotalSales",
    y="ProductName",
    orientation="h",
    text="GrandTotalSales",
    title="Top 10 Products by Total Sales"
)

fig.update_traces(textposition="outside")
fig.update_layout(
    xaxis_title="Total Sales Amount",
    yaxis_title="Product Name"
)

fig.show()

In [0]:
top_products = (
    spark.table("adbrag.project4_schema.gold_product_year_sales")
    .groupBy("ProductName")
    .agg(sum("TotalSalesAmount").alias("GrandTotalSales"))
    .orderBy(col("GrandTotalSales").desc())
    .limit(5)
)

top_product_list = [row["ProductName"] for row in top_products.collect()]

df_product_year = (
    spark.table("adbrag.project4_schema.gold_product_year_sales")
    .filter(col("ProductName").isin(top_product_list))
)

pdf_product_year = df_product_year.toPandas()

fig = px.bar(
    pdf_product_year,
    x="SourceYear",
    y="TotalSalesAmount",
    color="ProductName",
    barmode="group",
    title="Top Products Sales by Year"
)

fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Total Sales Amount"
)

fig.show()